In [61]:
from requests import get, post
from sys import exit
from pprint import pprint
import time

In [62]:
HOST = "https://api.chartmetric.com"
TOKEN = "eVCH88EH3jriTh1ps2OpDtXeDMwlPPM7GY9y2J8jFkBY3RuLtx19DIXzC1sn0Ygm"

res = post(f"{HOST}/api/token", json={"refreshtoken": TOKEN})
if res.status_code != 200:
    print(f"ERROR: received a {res.status_code} instead of 200 from /api/token")
    exit(1)

access_token = res.json()["token"]

def Get(endpoint, params=None):
    res = get(
        f"{HOST}{endpoint}",
        headers={"Authorization": f"Bearer {access_token}"},
        params=params
    )
    if res.status_code != 200:
        print("ERROR:", res.status_code, res.text)
        return None
    return res.json()

In [27]:
genres = Get("/api/genre")
genres

{'obj': [{'id': 501120, 'name': 'pop'},
  {'id': 501121, 'name': 'hip-hop/rap'},
  {'id': 501122, 'name': 'rock'},
  {'id': 501123, 'name': 'metal'},
  {'id': 501124, 'name': 'electronic'},
  {'id': 501125, 'name': 'r&b/soul'},
  {'id': 501126, 'name': 'classical'},
  {'id': 501127, 'name': 'jazz'},
  {'id': 501128, 'name': 'blues'},
  {'id': 501129, 'name': 'devotional & spiritual'},
  {'id': 501130, 'name': 'folk'},
  {'id': 501131, 'name': 'country'},
  {'id': 501132, 'name': 'alternative'},
  {'id': 501133, 'name': 'eras'},
  {'id': 501134, 'name': 'ambient'},
  {'id': 501135, 'name': 'psychedelic'},
  {'id': 501136, 'name': 'spoken / comedy'},
  {'id': 501137, 'name': 'vocal'},
  {'id': 501138, 'name': 'singer/songwriter'},
  {'id': 501139, 'name': 'soundtrack'},
  {'id': 501140, 'name': 'instrumental'},
  {'id': 501141, 'name': 'holiday'},
  {'id': 501142, 'name': 'easy listening'},
  {'id': 501143, 'name': 'worldbeat'},
  {'id': 501144, 'name': 'industrial'},
  {'id': 501145, 'n

In [28]:
# CONSTANTS
REACH_MAP = {
    "1": ("Undiscovered",  0,           100_000),
    "2": ("Developing",    100_000,     500_000),
    "3": ("Mid-level",     500_000,   2_000_000),
    "4": ("Mainstream",  2_000_000,  10_000_000),
    "5": ("Superstar",  10_000_000,  50_000_000),
    "6": ("Legendary",  50_000_000, 999_000_000),
}

AGE_KEY_MAP = {
    "13_17": "ages_13_17",
    "18_24": "ages_18_24",
    "25_34": "ages_25_34",
    "35_44": "ages_35_44",
    "45_64": "ages_45_64",
    "65_":   "ages_65_",
}

metric = "sp_monthly_listeners"

In [29]:
# INPUT HELPERS

def prompt_list(prompt, default=None):
    raw = input(f"{prompt} [{', '.join(default) if default else 'skip'}]: ").strip()
    if not raw and default:
        return default
    return [x.strip() for x in raw.split(",") if x.strip()]

def prompt_int(prompt, default):
    raw = input(f"{prompt} [{default}]: ").strip()
    return int(raw) if raw else default

# MOVIE DEMPOGRAPHICS PLACEHOLDER 

def get_movie_demographics(movie_title):
    return {
        "gender": "female",
        "age": ["18_24", "25_34"],
    }

# API CALLS

def get_audience_data(artist_id, audience_country, desired_interests):
    return {
        "stat": Get(f"/api/artist/{artist_id}/social-audience-stats", params={
            "domain": "instagram", "audienceType": "followers", "statsType": "stat"
        }),
        "demographic": Get(f"/api/artist/{artist_id}/social-audience-stats", params={
            "domain": "instagram", "audienceType": "followers", "statsType": "demographic"
        }),
        "country": Get(f"/api/artist/{artist_id}/social-audience-stats", params={
            "domain": "instagram", "audienceType": "followers", "statsType": "country"
        }) if audience_country else None,
        "interest": Get(f"/api/artist/{artist_id}/social-audience-stats", params={
            "domain": "instagram", "audienceType": "followers", "statsType": "interest"
        }) if desired_interests else None,
    }

In [30]:
# DEMOGRAPHIC FILTER

def matches_demographics(artist, gender, min_gender_total, age, min_age_total,
                          audience_country, min_audience_total, desired_interests):
    artist_id   = artist.get("chartmetric_artist_id") or artist.get("id")
    artist_name = artist.get("name", "Unknown")

    all_data = get_audience_data(artist_id, audience_country, desired_interests)

    # Total followers
    stat = all_data["stat"]
    if not stat or not stat.get("obj"):
        print(f"  ERROR: No stat data for {artist_name}")
        return False, {}
    total_followers = stat["obj"][0].get("followers", 0)
    if not total_followers:
        print(f"  ERROR: No follower count for {artist_name}")
        return False, {}
    print(f"  Followers: {total_followers:,}")

    # Gender Check
    demo = all_data["demographic"]
    if not demo or not demo.get("obj"):
        print(f"  ERROR: No demographic data for {artist_name}")
        return False, {}
    obj          = demo["obj"][0]
    gender_total = round((obj.get(gender) or 0) * total_followers)
    if gender_total < min_gender_total:
        return False, {"reason": f"{gender} total {gender_total:,} < min {min_gender_total:,}", "total_followers": total_followers}
    print(f"  {gender}: {gender_total:,}")

    # Age Check
    age_totals = {}
    for bucket in age:
        age_key = AGE_KEY_MAP.get(bucket)
        if not age_key:
            print(f"  ERROR: Age bucket '{bucket}' not recognized.")
            return False, {}
        age_totals[bucket] = round(obj.get(age_key, 0) * total_followers)
    if not all(age_totals[b] >= min_age_total for b in age):
        failing = {b: age_totals[b] for b in age if age_totals[b] < min_age_total}
        return False, {"reason": f"age buckets below threshold: {failing}", "total_followers": total_followers}
    for bucket in age:
        print(f"  {bucket}: {age_totals[bucket]:,}")

    # Audience Country Check
    country_totals = {}
    if audience_country:
        country_obj = all_data["country"]
        if not country_obj or not country_obj.get("obj"):
            print(f"  ERROR: No country data for {artist_name}")
            return False, {}
        entries = country_obj["obj"]
        for code in audience_country:
            match = next((e for e in entries if e.get("code2", "").upper() == code.upper()), None)
            country_totals[code] = round(match.get("weights", 0) * total_followers) if match else 0
        if not all(country_totals[c] >= min_audience_total for c in audience_country):
            failing = {c: country_totals[c] for c in audience_country if country_totals[c] < min_audience_total}
            return False, {"reason": f"countries below threshold: {failing}", "total_followers": total_followers}
        for code in audience_country:
            print(f"  {code}: {country_totals[code]:,}")

    # Interests Check
    if desired_interests:
        interest_obj = all_data["interest"]
        if not interest_obj or not interest_obj.get("obj"):
            print(f"  ERROR: No interest data for {artist_name}")
            return False, {}
        interest_names = [e.get("interest_name", "").lower() for e in interest_obj["obj"]]
        matched = [i for i in desired_interests if i.lower() in interest_names]
        print(f"  Interests matched: {matched if matched else 'none'}")

    return True, {
        "total_followers": total_followers,
        "gender_total":    gender_total,
        "age_totals":      age_totals,
        "country_totals":  country_totals if audience_country else {},
    }

In [31]:
# FILTER AND DISPLAY

def filter_and_display(artists_raw, gender, min_gender_total, age, min_age_total,
                        audience_country, min_audience_total, desired_interests,
                        desired_genres, genre_match_mode, desired_moods, mood_match_mode):

    # Genre/mood filter
    def matches_genre_mood(artist):
        genres_raw = artist.get("genres", "")
        if isinstance(genres_raw, str):
            tags = [t.strip().lower() for t in genres_raw.split(",")]
        else:
            tags = []
        mood_fn  = all if mood_match_mode  == "all" else any
        genre_fn = all if genre_match_mode == "all" else any
        mood_match  = mood_fn(m.lower()  in tags for m in desired_moods)  if desired_moods  else True
        genre_match = genre_fn(g.lower() in tags for g in desired_genres) if desired_genres else True
        return mood_match and genre_match

    artists_filtered = [a for a in artists_raw if matches_genre_mood(a)]
    print(f"Genre/mood filter: {len(artists_filtered)} artists remaining.\n")

    # Demographics filter
    matched_artists = []
    for artist in artists_filtered:
        name = artist.get("name", "Unknown")
        print(f"Checking: {name}...")
        is_match, result = matches_demographics(
            artist, gender, min_gender_total, age, min_age_total,
            audience_country, min_audience_total, desired_interests
        )
        if is_match:
            matched_artists.append({**artist, "demographics": result})
            print(f"  ✅ MATCH")
        else:
            if result:
                print(f"  No match — {result.get('reason', 'unknown reason')}")
                print(f"     total_followers: {result.get('total_followers', 'N/A')}")
        time.sleep(0.3)

    # Results
    print(f"\n── {len(matched_artists)} artists matched your criteria ──\n")
    for a in matched_artists:
        d = a["demographics"]
        print(f"🎵 {a['name']} ({(a.get('code2') or '?').upper()})")
        print(f"   Spotify listeners:  {(a.get('spotify_monthly_listeners') or a.get('sp_monthly_listeners') or 0):,}")
        if a.get("career_stage"):
            print(f"   Career stage:       {a.get('career_stage')}")
        if a.get("recent_momentum"):
            print(f"   Momentum:           {a.get('recent_momentum')}")
        if a.get("similarity"):
            print(f"   Similarity score:   {a.get('similarity'):.2f}")
        print(f"   Total IG followers: {d['total_followers']:,}")
        print(f"   {gender.capitalize()} followers:   {d['gender_total']:,}")
        for bucket, total in d["age_totals"].items():
            print(f"   Age {bucket}:         {total:,}")
        for code, total in d["country_totals"].items():
            print(f"   Audience in {code}:   {total:,}")
        print(f"   Genres: {a.get('genres', 'N/A')}")
        print()

In [32]:
# MODE A - SEARCH BASED ON AUDIENCE DEMOGRAPHICS

def run_mode_a():
    print()
    print("── Mode A: Search Based on Audience Demographics ──")

    prefilled = {}
    use_movie = input("\n  Start from a reference movie? (y / n): ").strip().lower()
    if use_movie == "y":
        movie_title = input("  Enter movie title: ").strip()
        print(f"  Fetching audience data for '{movie_title}'...")
        prefilled = get_movie_demographics(movie_title)
        print(f"  Prefilled — gender: {prefilled['gender']}, age: {prefilled['age']}")

    print("\n── Audience Demographics ──")
    gender             = input(f"  Target gender (female / male) [{prefilled.get('gender', 'female')}]: ").strip().lower() or prefilled.get("gender", "female")
    min_gender_total   = prompt_int("  Min absolute followers of that gender", 50_000)
    age                = prompt_list("  Age buckets (13_17, 18_24, 25_34, 35_44, 45_64, 65_)", prefilled.get("age", ["18_24", "25_34"]))
    min_age_total      = prompt_int("  Min absolute followers per age bucket", 10_000)
    audience_country   = prompt_list("  Audience countries (e.g. AR, PE, US)", [])
    min_audience_total = prompt_int("  Min absolute followers per country", 100_000) if audience_country else 0
    desired_interests  = prompt_list("  Desired interests (e.g. fashion, beauty)", [])

    print("\n── Music Filters ──")
    desired_genres   = prompt_list("  Genres (e.g. latin pop, reggaeton)", [])
    genre_match_mode = input("  Genre match mode (any / all) [any]: ").strip().lower() or "any"
    desired_moods    = prompt_list("  Moods (e.g. celebratory, sad)", [])
    mood_match_mode  = input("  Mood match mode (any / all) [any]: ").strip().lower() or "any"

    print("\n── Artist Filters ──")
    artist_country = prompt_list("  Artist countries (e.g. AR, CL, US)", [])
    print("  Artist reach:")
    for key, (label, mn, mx) in REACH_MAP.items():
        print(f"    {key}. {label} ({mn:,} – {mx:,} monthly listeners)")
    reach_choice        = input("  Select reach [3]: ").strip() or "3"
    _, min_pop, max_pop = REACH_MAP.get(reach_choice, REACH_MAP["3"])
    limit               = prompt_int("  Max artists to fetch per country", 200)

    # Fetch artists
    print("\n🔍 Fetching artists...\n")
    artists_raw = []
    for code in artist_country:
        params = {"min": min_pop, "max": max_pop, "code2": code, "limit": limit}
        res = Get(f"/api/artist/{metric}/list", params=params)
        if res and "obj" in res:
            batch = res["obj"]["data"]
            artists_raw.extend(batch)
            print(f"  {code}: {len(batch)} artists found")
        else:
            print(f"  {code}: ERROR retrieving artists")
    seen = set()
    artists_raw = [
        a for a in artists_raw
        if not (a["chartmetric_artist_id"] in seen or seen.add(a["chartmetric_artist_id"]))
    ]
    print(f"\n{len(artists_raw)} unique artists found.\n")

    filter_and_display(
        artists_raw, gender, min_gender_total, age, min_age_total,
        audience_country, min_audience_total, desired_interests,
        desired_genres, genre_match_mode, desired_moods, mood_match_mode
    )

In [ ]:
# MODE B — FIND SIMILAR ARTISTS

def get_artist_id(name):
    res = Get("/api/search", params={"q": name, "type": "artists", "limit": 1})
    if not res or not res.get("obj") or not res["obj"].get("artists"):
        print(f"  ERROR: Could not find artist '{name}'")
        return None
    artist = res["obj"]["artists"][0]
    print(f"  Found: {artist['name']} (id: {artist['id']}, listeners: {(artist['sp_monthly_listeners'] or 0):,})")
    return artist["id"]

def get_similar_artists(artist_id, audience=None, mood=None, genre=None, musicality=None, limit=10):
    params = {"limit": limit}
    if audience:   params["audience"]   = audience
    if mood:       params["mood"]       = mood
    if genre:      params["genre"]      = genre
    if musicality: params["musicality"] = musicality
    return Get(f"/api/artist/{artist_id}/similar-artists/by-configurations", params=params)

def run_mode_b(artist_names, audience_sim=None, mood_sim=None,
               genre_sim=None, musicality_sim=None,
               desired_reach=None, artist_country_filter=None,
               gender=None, min_gender_total=0,
               age=None, min_age_total=0,
               audience_country=None, min_audience_total=0,
               desired_interests=None,
               desired_genres=None, genre_match_mode="any",
               desired_moods=None, mood_match_mode="any",
               limit=50):

    print("\n── Mode B: Find Similar Artists ──")

    # Resolve names to IDs
    print("\nResolving artist names...")
    featured_ids = []
    for name in artist_names:
        artist_id = get_artist_id(name)
        if artist_id:
            featured_ids.append(artist_id)
        time.sleep(0.3)

    if not featured_ids:
        print("ERROR: No artists could be resolved.")
        return

    if desired_reach:
        _, min_pop, max_pop = REACH_MAP.get(desired_reach, REACH_MAP["3"])
    else:
        min_pop, max_pop = 0, 999_000_000

    # Fetch similar artists
    print(f"\nFetching similar artists...\n")
    all_similar = {}

    for artist_id in featured_ids:
        res = get_similar_artists(artist_id, audience_sim, mood_sim, genre_sim, musicality_sim, limit)
        if not res or "obj" not in res:
            print(f"  ERROR: No similar artists found for id {artist_id}")
            continue

        batch = res["obj"]["data"]
        added = 0
        
        for a in batch:
            aid = a.get("id")

            if not aid or aid in all_similar or aid in featured_ids:
                continue

            # Popularity Filter
            listeners = a.get("sp_monthly_listeners")

            if listeners is not None:
                if not (min_pop <= listeners <= max_pop):
                    continue

            # Artist Country Filter
            if artist_country_filter:
                code = a.get("code2")
                if code:
                    if code.upper() not in {c.upper() for c in artist_country_filter}:
                        continue
                else:
                    pass

            # Passed all filters
            all_similar[aid] = a
            added += 1

    artists_raw = sorted(all_similar.values(), key=lambda x: x.get("similarity", 0), reverse=True)
    print(f"\n{len(artists_raw)} unique similar artists found before demographic filter.\n")

    if not artists_raw:
        print("No artists passed pre-filter. Try relaxing desired_reach or artist_country_filter.")
        return

    filter_and_display(
        artists_raw, gender, min_gender_total, age, min_age_total,
        audience_country, min_audience_total, desired_interests,
        desired_genres, genre_match_mode, desired_moods, mood_match_mode
    )

In [34]:
"""
LANDING SCREEN
"""

def run_syncfit():
    print("         🎵 SyncFit — Artist Discovery")
    print()
    print("How would you like to find artists?")
    print()
    print("  A — Search Based on Audience Demographics")
    print("      Search by audience demographics, genre, mood and popularity.")
    print("      You can use a reference movie's audience as your starting point.")
    print()
    print("  B — Find Similar Artists")
    print("      Find artists similar to those featured in a reference movie.")
    print()

    while True:
        choice = input("Select a mode (A / B): ").strip().upper()
        if choice in ["A", "B"]:
            break
        print("  ERROR: Invalid choice, please enter A or B.")

    print()
    if choice == "A":
        run_mode_a()
    elif choice == "B":
        run_mode_b()

In [35]:
#run_syncfit()

In [37]:
# TESTING WITH HARDCODED PROMPTS

# Test Mode A
gender             = "male"
min_gender_total   = 100_000
age                = ["18_24", "25_34"]
min_age_total      = 50_000
audience_country   = ["MX"]
min_audience_total = 50_000
desired_interests  = []

desired_genres     = ["rock", "r&b/soul", "funk"]
genre_match_mode   = "any"
desired_moods      = []
mood_match_mode    = "any"

artist_country     = ["MX"]
_, min_pop, max_pop = REACH_MAP["4"]
limit              = 200

# Fetch artists
artists_raw = []
for code in artist_country:
    params = {"min": min_pop, "max": max_pop, "code2": code, "limit": limit}
    res = Get(f"/api/artist/{metric}/list", params=params)
    if res and "obj" in res:
        batch = res["obj"]["data"]
        artists_raw.extend(batch)
        print(f"  {code}: {len(batch)} artists found")
seen = set()
artists_raw = [
    a for a in artists_raw
    if not (a["chartmetric_artist_id"] in seen or seen.add(a["chartmetric_artist_id"]))
]
print(f"\n{len(artists_raw)} unique artists found.\n")

filter_and_display(
    artists_raw, gender, min_gender_total, age, min_age_total,
    audience_country, min_audience_total, desired_interests,
    desired_genres, genre_match_mode, desired_moods, mood_match_mode
)

  MX: 200 artists found

200 unique artists found.

Genre/mood filter: 18 artists remaining.

Checking: Intocable...
  Followers: 771,869
  male: 289,168
  18_24: 153,147
  25_34: 369,355
  MX: 467,001
  ✅ MATCH
Checking: Caifanes...
  Followers: 621,796
  male: 297,536
  18_24: 178,076
  25_34: 266,418
  MX: 431,558
  ✅ MATCH
Checking: Los Plebes del Rancho de Ariel Camacho...
  Followers: 1,606,300
  male: 710,340
  18_24: 694,999
  25_34: 670,450
  MX: 1,150,562
  ✅ MATCH
Checking: Yuridia...
  Followers: 1,652,717
  male: 326,235
  18_24: 361,928
  25_34: 836,172
  MX: 1,105,782
  ✅ MATCH
Checking: El Fantasma...
  Followers: 3,288,177
  male: 1,956,945
  18_24: 1,148,534
  25_34: 1,519,881
  MX: 1,652,559
  ✅ MATCH
Checking: Armenta...
  Followers: 136,296
  No match — male total 95,749 < min 100,000
     total_followers: 136296
Checking: Gibran Alcocer...
ERROR: 429 Rate limit exceeded, retry in -172 ms
  ERROR: No stat data for Gibran Alcocer
Checking: Inspector...
  Followers: 

In [66]:
# Test Mode B - Pulp Fiction
artist_names = [
    "Al Green",
    "Brothers Johnson",
    "Chuck Berry",
    "Dick Dale & His Del-Tones",
    "Dusty Springfield",
    "Kool & The Gang",
    "Link Wray",
    "Maria McKee",
    "Ricky Nelson",
    "The Centurians",
    "The Lively Ones",
    "The Marketts",
    "The Revels",
    "The Robins",
    "The Statler Brothers",
    "The Tornadoes",
    "Urge Overkill",
    "Woody Thorne",
]

run_mode_b(
    artist_names          = artist_names,
    genre_sim             = "medium",
    musicality_sim        = "medium",
    desired_reach         = "1",
    artist_country_filter = ["US"],
    gender                = "male",
    min_gender_total      = 500,
    age                   = ["18_24", "25_34"],
    min_age_total         = 500,
    audience_country      = ["US"],
    min_audience_total    = 1_000,
    desired_interests     = [],
    desired_genres        = [],
    genre_match_mode      = "any",
    desired_moods         = [],
    mood_match_mode       = "any",
    limit                 = 10
)


── Mode B: Find Similar Artists ──

Resolving artist names...
  Found: Al Green (id: 365, listeners: 7,488,125)
  Found: The Brothers Johnson (id: 209108, listeners: 1,282,877)
  Found: Chuck Berry (id: 373, listeners: 21,824,183)
  Found: Dick Dale & His Del-Tones (id: 472690, listeners: 434,805)
  Found: Dusty Springfield (id: 17, listeners: 5,953,860)
  Found: Kool & The Gang (id: 208947, listeners: 9,584,327)
  Found: Link Wray (id: 189208, listeners: 493,505)
  Found: Maria McKee (id: 194051, listeners: 476,777)
  Found: Ricky Nelson (id: 212015, listeners: 2,101,050)
  Found: The Centurians (id: 482397, listeners: 152,129)
  Found: The Lively Ones (id: 212018, listeners: 293,036)
  Found: The Marketts (id: 163567, listeners: 105,073)
  Found: The Revels (id: 59765, listeners: 16,551)
  Found: The Robins (id: 55614, listeners: 30,954)
  Found: The Statler Brothers (id: 212017, listeners: 522,857)
  Found: The Tornadoes (id: 215347, listeners: 55,151)
  Found: Urge Overkill (id: 2